# Dreamer V1 from Scratch

In [1]:
import torch
from torch import nn
import torch.nn.functional as F

### Markov Decision Process (MDP)

Markov Systems are systems where the next state depends only on the current state and the action taken

A MDP models how the state of a system evolves when different actions are applied to the system.

Let:
- $S$ be the set of states in a system
- $A$ be the set of potential actions; they may modify the state of the actor, i.e. change his state to another one within $S$
- $T : (S \times A \times S) \rightarrow [0, 1]$ be a *transition function* such that $T(s, a, s') = P(s' | s, a)$ (i.e. the probability of reaching state $s'$ given that action $a$ is applied when the actor has the state $s$). Since $T$ is a probability distribution, we have that $\sum_{s' \in S} T(s, a, s') = 1$
- $r : S \times A \rightarrow \R$ be the reward function such that the actor gets reward $r(s, a)$ if it takes action $a$ at state $s$. If the reward is small, then the action is less useful for achieving the goal

Then, we define $MDP : (S, A, T, R, \gamma), \text{where } \gamma \in [0, 1)$ is the discount factor

Reward is calculated as $R(\tau) = r_0 + r_1 + r_2 + ...$ for a trajectory $\tau$, and there may be a discount factor $\gamma$ such that $R(\tau) = r_0 + \gamma r_1 + \gamma^2 r_2 + ...$

The ultimate goal is to maximize $R(\tau)$

[Bibliography](https://d2l.ai/chapter_reinforcement-learning/mdp.html)

### Partially Observable MDP

Imagine playing a video game where you can only see the image (frame). Then, since that is the only information available to you, there is no way of tracking the exact state parameters the game state registers, such as velocity and position. Thus, it breaks the Markov System presumption that given a state and an action, the next state can be reliably predicted.

In this case, we are dealing with an observation $o_t$ which holds only partial or potentially noisy information regarding the state $s_t$.

Thus, we are dealing with a POMDP when we have:
- A hidden space state $S$
- An observation space $O$ and a function which maps states to observations
- Everything else from a MDP

The approach is to create a belief state distribution $b_t = P(o_\text{1:t}, a_\text{1, t-1})$ over what the true state might be, and we update it with every new observation.

### Reinforcement Learning Agent

$a_t \sim p(a_t | o_\text{1:t}, a_\text{1, t-1})$ and $o_t, r_t \sim p(o_t, r_t | o_\text{1:t-1}, a_\text{1, t-1})$

*The goal is to develop an agent that maximizes the expected sum of the rewards* $\mathbb E_p(\sum_\text{t=1}r_t)$

### The Value Function
The expected cumulative discounted reward you'll collect starting from state $s$ and following policy $\pi$ forever.

$$V^\pi(s) = \mathbb{E}\left[\sum_{t=0}^{\infty} \gamma^t r_t \;\middle|\; s_0 = s, \pi\right]$$

Basically, it says: *if I'm in state $s$ and follow policy $\pi$ forever, how much total discounted reward will I collect?* 

**Claude** says:
    *Its goal is to compress an infinite future into a single number that you can reason about. Different policies produce different value functions — a bad policy has low values everywhere, a good one has high values.*

The optimal policy $\pi^*$ is the one whose value function is maximal at every state simultaneously:

$$V^*(s)=\max⁡_\pi V^\pi(s) \;\forall s$$

### The Bellman Equation

We separate the current state from the rest of the states:
$$V^\pi(s) = r(s, \pi(s)) + \gamma \, \mathbb{E}_{s' \sim P(\cdot \mid s, \pi(s))}\left[V^\pi(s')\right]$$

Therefore, we get something like: *the value of where you are = what you get right now + discounted average value of where you'll end up*.

This is a reformulation of the value function, which reveals a recursive form that we can use for computation (can't compute an infinite sum), since $s'$ is the state immediately following $s$.

Its goal is to give you a checkable consistency condition, i.e. if your estimate of $V_\pi$ is correct, this equation must hold everywhere. If it doesn't hold somewhere, you have a concrete, computable error. This is the crucial bridge to learning.

For a finite state space, this is literally a system of linear equations (one per state). Because $0 <= \gamma < 1$, the matrix $(I−\gamma P)$ is always invertible, since $\gamma < 1$ and every eigenvalue of $P$ is smaller or equal to 1 (property of stochastic matrices; each row is a probability distribution), so all eigenvalues of $(I−\gamma P)$ are greater than 0 (so the matrix is invertible). Thus, it guarantees a unique solution: the true $V_\pi$. This uniqueness is what makes it a well-defined target.



### TD Learning (Temporal Difference Learning)

In practice, you don't know $P$ or $r$ analytically, so you can't solve the linear system directly. Instead, you sample experience and iteratively enforce the Bellman equation one step at a time.

After observing one real transition $(s,a,r,s′)$, you compute the TD target, i.e. what the Bellman equation says $V(s)$ should approximately equal to: $$\text{target} = r + \gamma \hat{V}(s′)$$

And nudge your estimate toward it:
$$\hat{V}(s) \leftarrow \hat{V}(s) + \alpha [r + \gamma \hat{V}(s′) − \hat{V}(s)]$$

We don't have access to the true $V_\pi$. The TD $\text{target} = r + \gamma \hat{V}(s′)$ is not the truth, but a bootstrap.
- $\hat(V)(s)$ is the current estimate at the state you just left
- $\gamma \hat{V}(s′)$ is a slightly better-grounded estimate, because it anchors to one real reward $r$ before using the approximation

Instead of targeting a well-known result, we're chasing a moving target that is itself an approximation. 

The term in brackets is the TD error, which calculates how much your current estimate violates the Bellman equation at this state. The Bellman operator is a contraction mapping, meaning that each application shrinks the gap between your estimate and the truth by a factor of $\gamma$*.


*) **Claude** explains:

*At every update, your estimate at $(s_2)$ is:*

$$[\hat{V}^{(n+1)}(s_2) = 1 + 0.9 \times \hat{V}^{(n)}(s_2)]$$

*The true value satisfies the same equation exactly:*

$$[V^*(s_2) = 1 + 0.9 \times V^*(s_2)]$$

*Subtract one from the other:*

$$[V^*(s_2) - \hat{V}^{(n+1)}(s_2) = 0.9 \times \left(V^*(s_2) - \hat{V}^{(n)}(s_2)\right)]$$

*The gap at step $(n+1)$ is exactly 0.9 times the gap at step $n$. Always. The reward term cancels, so the only thing that survives is $\gamma$ multiplying the old gap.*

| Component           | Goal                                                                         |
| ------------------- | ---------------------------------------------------------------------------- |
| MDP                 | Formally defines the environment: states, actions, transitions, rewards |
| Value function      | Measures how good a state is under a given policy |
| Bellman equation    | Characterizes the correct value function via a local consistency condition |
| TD learning         | Enforces that consistency from sampled data, without knowing the environment |
| Policy optimization | Uses the learned value function to improve the policy towards $\pi^*$ (optimal policy) |

### The Actor

The actor is the policy $\pi$. It takes a state $s$ as input and outputs an action $a$. More precisely, a probability distribution over potential actions. Its goal is to find the policy that maximizes $V_\pi (s)$.

In Dreamer specifically, the actor takes a latent state as input and outputs which action to take in the real environment.

### The Critic

The critic is the value function $V_\pi$. It takes a state as input and outputs a single number, the expected future reward from that state under the current policy. It doesn't decide what to do; it just evaluates how good the current situation is.​

The critic is trained using TD learning to enforce Bellman consistency.

### The Flow
1. Actor outputs action $a$ given state $s$
2. Environment (or world model) produces $r$ and $s′$
3. Critic evaluates both $s$ and $s′$, computes TD error
4. TD error flows back to update the critic (improve value estimates) and the actor (improve action choices)


### Latent Dynamics

We use $p$ to denote models which have access to real observations $o_t$ and $q$ to denote models which have inputs solely from latent space.
- $s_t$ is the state in latent space
- $a_t$ is the action
- $o_t$ is the observation
- $q_t$ is the reward

There are three models inside the Dreamer architecture:
1. The representation model: $p(s_t | s_\text{t-1}, a_\text{t-1}, o_\text{t})$
2. The transition model: $q(s_t | s_\text{t-1}, a_\text{t-1})$
3. The reward model: $q(r_t | s_t)$

### Action and Value Models

1. The action model: $a_\tau \sim q_\phi(a_\tau | s_\tau)$
2. The value model: $v_\psi(s_t) \approx \mathbb E_{q(\cdot| s_t)} \left[ \sum_{\tau = t}^{t + H} \gamma^{\tau - t} r_\tau \right]$ 

We sample the action from the distribution like so:
$$a_\tau = \tanh(\mu_\phi(s_\tau) + \sigma_\phi(s_\tau) \epsilon), \quad \epsilon \sim \text{Normal}(0, I)$$

We use $\epsilon$ for the sample so that we can then treat it as a constant and backpropagate.

### Value Estimation

The reasoning behind this is mixing short-horizon returns (low variance, more bias) and long-horizon returns (low bias, more variance) to get a good target for training the critic.

Parameters:

- $\gamma$: discount factor (typically 0.99)
- $\lambda$: trace decay (typically 0.95)
- $H$: imagination horizon (typically 15)
- $v_\psi$​: learned value function

$$
\text{k-Step Return: } \quad V^k_N(s_\tau) = \mathbb{E}\left[\sum_{n=\tau}^{h-1} \gamma^{n-\tau} r_n + \gamma^{h-\tau} v_\psi(s_h)\right] \text{where } h = \min(\tau + k, t+H)
$$

$$
\lambda \text{-Return: } \quad V^\lambda(s_\tau) = (1-\lambda)\sum_{n=1}^{H-1} \lambda^{n-1} V^n_N(s_\tau) + \lambda^{H-1} V^H_N(s_\tau)
$$

For k-Step Return: From $t$ to $t + H$ we use the actual rewards, that we actually compute and we use an exponentially weighted average. Then, from $t + H$ on, we use our value estimation model. It has a similar structure to that of a Taylor expansion.

For $\lambda$-Return: Each one is a DIFFERENT estimate of the same thing! They all try to estimate $V(s_t)$ but use different amounts of real data vs bootstrapping.


The $(1 - \lambda)$ factor is a normalization trick to make sure the sum's $\lambda$ parameters sum up to 1. For the same reason, the last term is pulled out of the sum.

### Learning Objective

1. Fix the world model
2. Roll out H steps in latent space with the current actor
3. From this rollout, compute $\lambda$-Return targets $V_\lambda(s_\tau)$, using the value net
4. Update:
    - Critic: train $v_\psi(s_\tau)$ to get as close as possible to $V_\lambda(s_\tau)$
    - Actor: backpropagate gradients through $v_\psi$, reward model, and dynamics to choose actions that lead to high $V_\lambda$

Note: $V_\lambda$ is more stable and lower-variance than a single n-step return. Nevertheless, it still depends on the current value network at the tail, so updating $v_\psi$ toward $V_\lambda$ is a form of policy evaluation by bootstrapping.
 

### Learning Latent Dynamics

What training objective do we use to make these latent states useful for control?
- Reward prediction: only care that the latent state is good enough to predict reward
- Reconstruction: learn to reconstruct images and rewards from the latent state
- Contrastive* estimation: instead of predicting pixels, use a contrastive objective so that states are predictable from images


*) Contrastive: separate the true pair from many impostors to learn a representation where the right matches are easy to pick out

### RSSM

RSSM is Dreamer’s world model, basically an RNN with two heads. It keeps an internal state that can be updated with actions and then updated to imagine future trajectories.

At each time step, the state has two parts:  
  - $(h_t)$: deterministic memory (RNN hidden) 
  - $(z_t)$: stochastic latent (Gaussian-sampled vector, like a VAE latent, but per time step)

Two small “heads” output distributions over $z_t$:  
  - Prior head: uses only $h_t$; used when imagining the future without images
  - Posterior head: uses $h_t + o_t$; used during training when images are available

Given $h_t$, the **prior** over $z_t$ is a Gaussian:
$$
p_\theta(z_t \mid h_t) = \mathcal{N}\big(\mu^\text{prior}_t, \Sigma^\text{prior}_t\big)
$$

During training, when the image $o_t$ is available, the **posterior** is:
$$
q_\theta(z_t \mid h_t, o_t) = \mathcal{N}\big(\mu^\text{post}_t, \Sigma^\text{post}_t\big)
$$

Training samples $z_t$ from the posterior, reconstructs observation and reward from $(h_t, z_t)$, and adds a KL term to push posterior and prior closer so the prior works well during imagination.


### Architecture

In [2]:
class RSSM(nn.Module):
    def __init__(self, latent_size, action_size, hidden_size, obs_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.latent_size = latent_size
        self.action_size = action_size
        self.gru = nn.GRUCell(latent_size + action_size, hidden_size)
        self.prior = nn.Linear(hidden_size, 2 * latent_size)
        self.posterior = nn.Linear(hidden_size + obs_size, 2 * latent_size)

    def init_state(self, batch_size, device):
        h = torch.zeros(batch_size, self.hidden_size, device=device)
        z = torch.zeros(batch_size, self.latent_size, device=device)
        return h, z

    def imagine_step(self, h, z_t, a_t):
        # [B, latent_size + action_size]
        x = torch.cat((z_t, a_t), dim=-1)
        # [B, hidden_size]
        h = self.gru(x, h)
        # [B, 2 * latent_size]
        out_prior = self.prior(h)
        mean, log_std = out_prior.chunk(2, dim=-1)
        std = torch.exp(log_std)
        eps = torch.randn_like(mean)
        z = mean + std * eps
        return h, z, (mean, std)

    def observe_step(self, h, z_t, a_t, o_t):
        # [B, latent_size + action_size]
        x = torch.cat((z_t, a_t), dim=-1)
        # [B, hidden_size]
        h = self.gru(x, h)
        # [B, 2 * latent_size]
        stats_prior = self.prior(h)
        mean_pri, log_std_pri = stats_prior.chunk(2, dim=-1)
        std_pri = torch.exp(log_std_pri)
        
        # [B, hidden_size + obs_size]
        hp = torch.cat((h, o_t), dim=-1)
        out_post = self.posterior(hp)
        mean_post, log_std_post = out_post.chunk(2, dim=-1)
        std_post = torch.exp(log_std_post)

        eps = torch.randn_like(mean_post)
        z = mean_post + std_post * eps

        prior = (mean_pri, std_pri)
        posterior = (mean_post, std_post)

        return h, z, prior, posterior   
        

In [3]:
class ObsEncoder(nn.Module):
    def __init__(self, obs_size, embed_size):
        super().__init__()

        self.obs_size = obs_size
        self.embed_size = embed_size

        self.mlp = nn.Sequential(
            nn.Linear(obs_size, 64),
            nn.ReLU(),
            nn.Linear(64, embed_size),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.mlp(x)


In [4]:
class ObsDecoder(nn.Module):
    def __init__(self, hidden_size, latent_size, obs_size):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size + latent_size, 64),
            nn.ReLU(),
            nn.Linear(64, obs_size)
        )
    
    def forward(self, z_t, h_t):
        x = torch.cat((z_t, h_t), dim=-1)
        return self.mlp(x)

In [5]:
class RewardDecoder(nn.Module):
    def __init__(self, hidden_size, latent_size):
        super().__init__()

        self.hidden_size = hidden_size
        self.latent_size = latent_size

        self.mlp = nn.Sequential(
            nn.Linear(hidden_size + latent_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, z_t, h_t):
        x = torch.cat((z_t, h_t), dim=-1)
        return self.mlp(x)


In [6]:
class Actor(nn.Module):
    def __init__(self, hidden_size, latent_size, action_size):
        super().__init__()

        self.hidden_size = hidden_size
        self.latent_size = latent_size
        self.action_size = action_size

        self.mlp = nn.Sequential(
            nn.Linear(hidden_size + latent_size, 64),
            nn.ReLU(),
            nn.Linear(64, action_size * 2),
            nn.ReLU()
        )
    
    def forward(self, z_t, h_t):
        # [B, latent_size + hidden_size]
        x = torch.cat((z_t, h_t), dim=-1)
        
        # [B, 2 * action_size]
        out = self.mlp(x)

        # [B, action_size]
        mean, log_std = out.chunk(2, dim=-1)
        std = torch.exp(log_std)
        eps = torch.randn_like(mean)

        # actions in [-1,1]
        a = F.tanh(mean + std * eps)
        return a

In [7]:
class Critic(nn.Module):
    def __init__(self, hidden_size, latent_size):
        super().__init__()

        self.hidden_size = hidden_size
        self.latent_size = latent_size

        self.mlp = nn.Sequential(
            nn.Linear(hidden_size + latent_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, z_t, h_t):
        x = torch.cat((z_t, h_t), dim=-1)
        return self.mlp(x)


In [8]:
class Dreamer(nn.Module):
    def __init__(self, obs_size, latent_size, hidden_size, action_size):
        super().__init__()
        self.obs_size = obs_size
        self.latent_size = latent_size
        self.hidden_size = hidden_size
        self.action_size = action_size

        self.rssm = RSSM(latent_size, action_size, hidden_size, obs_size)
        self.obs_encoder = ObsEncoder(obs_size, action_size)
        self.obs_decoder = ObsDecoder(hidden_size, latent_size, obs_size)
        self.reward_decoder = RewardDecoder(hidden_size, latent_size)
        self.actor = Actor(hidden_size, latent_size, action_size)
        self.critic = Critic(hidden_size, latent_size)

    def rollout_observe(self, obs_seq, act_seq, device):
        """
        obs_seq: [T, B, obs_size]
        act_seq: [T, B, action_size]
        Returns: lists of h_t, z_t, prior stats, post stats
        """
        T, B, _ = obs_seq.shape
        h, z = self.rssm.init_state(B, device)

        hs, zs, priors, posts = [], [], [], []

        for t in range(T):
            # [B, obs_embed_dim]
            o_t = self.obs_encoder(obs_seq[t])

            # [B, action_size]
            a_t = act_seq[t]
            
            h, z, prior, post = self.rssm.observe_step(h, z, a_t, o_t)
            hs.append(h)
            zs.append(z)
            priors.append(prior)
            posts.append(post)
        return hs, zs, priors, posts

    def rollout_imagine(self, h0, z0, H, device):
        """
        h0, z0: [B, hidden_size], [B, latent_size]
        H: imagination horizon
        Returns: lists of s_t, a_t, r_hat_t
        """
        h, z = h0, z0
        hs, zs, acts, rews = [], [], [], []

        for t in range(H):
            # [B, action_size]
            a_t, _ = self.actor(z, h)
            h, z, _ = self.rssm.imagine_step(h, z, a_t)
            
            # [B, 1]
            r_hat = self.reward_decoder(z, h)         

            hs.append(h)
            zs.append(z)
            acts.append(a_t)
            rews.append(r_hat)

        return hs, zs, acts, rews

### Reward Loss

This is a simple MSE Loss function

In [9]:
def reward_loss(dreamer, hs, zs, rew_seq):
    """
    hs, zs: lists of length T, each element [B, hidden_size] / [B, latent_size]
    rew_seq: [T, B] true rewards
    """
    T = len(hs)
    rew_seq = rew_seq.to(hs[0].device)

    losses = []
    for t in range(T):
        h_t = hs[t]
        z_t = zs[t]
        r_hat = dreamer.reward_decoder(z_t, h_t).squeeze(-1)   # [B]
        losses.append(F.mse_loss(r_hat, rew_seq[t], reduction="mean"))
    return sum(losses) / T

### KL Divergence

It computes the difference between the two Gaussian distributions, $N(\mu_{post}, \sigma_{post})$ and $N(\mu_{pri}, \sigma_{pri})$, the prior we predict from memory.

$$\text{Loss} = ||\text{input} - \text{reconstruction}||^2 + KL(N(\mu_{post}, \sigma_{post}) || N(\mu_{pri}, \sigma_{pri}))$$

The KL formula is self-correcting:

* $\mu_{post}$ drifts from $\mu_{pri}$ → KL penalizes it
* $\sigma_{post}$ drifts from $\sigma_{pri}$ → KL penalizes it

$$KL = \sum\left(\log\frac{\sigma_{pri}}{\sigma_{post}} + \frac{\sigma_{post}^2 + (\mu_{post} - \mu_{pri})^2}{2\sigma_{pri}^2} - 0.5\right)$$

In [10]:
def kl_loss(priors, posts):
    """
    priors, posts: lists of length T
      each element is (mean, std) with shape [B, latent_size]
    KL(q || p) for two gaussians, averaged over batch and time.
    """
    T = len(priors)
    kl_terms = []
    for t in range(T):
        mean_pri, std_pri = priors[t]
        mean_post, std_post = posts[t]

        var_pri = std_pri ** 2
        var_post = std_post ** 2

        kl = torch.log(std_pri / std_post) \
             + (var_post + (mean_post - mean_pri) ** 2) / (2.0 * var_pri) \
             - 0.5
        
        kl = kl.sum(dim=-1)
        kl = torch.clamp(kl, min=3.0)
        kl = kl.mean()
        kl_terms.append(kl)
    return sum(kl_terms) / T 

### Critic Loss

Again, we use MSE

In [11]:
def critic_loss(dreamer, hs, zs, targets):
    """
    hs, zs: lists length T, each [B, hidden_size]/[B, latent_size]
    targets: [T, B] lambda-return targets V_lambda(s_t)
    """
    T = len(hs)
    targets = targets.to(hs[0].device)

    vals = []
    for t in range(T):
        v_t = dreamer.critic(zs[t], hs[t]).squeeze(-1)  # [B]
        vals.append(v_t)
    vals = torch.stack(vals, dim=0)  # [T, B]

    return 0.5 * F.mse_loss(vals, targets, reduction="mean")

### Actor Loss

We want to maximize the targets, that is the $\lambda$-Returns

In [12]:
def actor_loss(targets):
    """
    targets: [T, B] lambda-return values for imagined states under current policy
    """
    return -targets.mean()

### Train World Model

In [13]:
def train_world_model_step(dreamer, wm_opt, replay, batch_len, batch_size, device, beta=1.0):
    obs_seq, act_seq, rew_seq = replay.sample_sequences(batch_len, batch_size)
    obs_seq = obs_seq.to(device)        # [T,B,obs_size]
    act_seq = act_seq.to(device)        # [T,B,action_size]
    rew_seq = rew_seq.to(device)        # [T,B]

    T, B, _ = obs_seq.shape

    # run RSSM in observer mode over the sequence
    h, z = dreamer.rssm.init_state(B, device)
    hs, zs, priors, posts = [], [], [], []

    for t in range(T):
        o_t = dreamer.obs_encoder(obs_seq[t])  # [B, embed_size]
        a_t = act_seq[t]                       # [B, action_size]
        h, z, prior, post = dreamer.rssm.observe_step(h, z, a_t, o_t)

        hs.append(h)
        zs.append(z)
        priors.append(prior)
        posts.append(post)

    rew_l = reward_loss(dreamer, hs, zs, rew_seq)
    obs_l = 0
    for t in range(T):
        o_hat = dreamer.obs_decoder(zs[t], hs[t])
        obs_l += F.mse_loss(o_hat, obs_seq[t])
    obs_l = obs_l / T

    kl_l = kl_loss(priors, posts)

    loss = obs_l + rew_l + beta * kl_l

    wm_opt.zero_grad()
    loss.backward()
    wm_opt.step()

    return loss.item(), rew_l.item(), kl_l.item()


### $\lambda$-Returns

In [14]:
def lambda_returns(rews, values, gamma=0.99, lam=0.95):
    T, B = rews.shape
    returns = torch.zeros_like(rews)
    next_value = values[-1]   # bootstrap

    for t in reversed(range(T)):
        delta = rews[t] + gamma * next_value - values[t]
        next_value = values[t] + lam * delta
        returns[t] = next_value
    return returns


### Train Behaviour

In [15]:
def sample_latent_starts(dreamer, replay, prefix_len, batch_size, device):
    obs_seq, act_seq, _ = replay.sample_sequences(prefix_len, batch_size)
    obs_seq = obs_seq.to(device)
    act_seq = act_seq.to(device)

    T, B, _ = obs_seq.shape
    h, z = dreamer.rssm.init_state(B, device)
    hs, zs = [], []

    for t in range(T):
        o_t = dreamer.obs_encoder(obs_seq[t])
        a_t = act_seq[t]
        h, z, _, _ = dreamer.rssm.observe_step(h, z, a_t, o_t)
        hs.append(h)
        zs.append(z)

    h0 = hs[-1].detach()
    z0 = zs[-1].detach()
    return h0, z0


In [16]:
def train_behavior_step(dreamer, actor_opt, critic_opt, replay, H, device, gamma=0.99, lam=0.95):
    h0, z0 = sample_latent_starts(dreamer, replay, prefix_len=10, batch_size=32, device=device)

    h = h0
    z = z0
    hs, zs, rews = [], [], []

    for t in range(H):
        # [B, action_size]
        a_t = dreamer.actor(z, h) 
        h, z, _ = dreamer.rssm.imagine_step(h, z, a_t)
        r_hat = dreamer.reward_decoder(z, h).squeeze(-1)  # [B]
        hs.append(h)
        zs.append(z)
        rews.append(r_hat)

    # list of length H
    hs_t = hs
    zs_t = zs
    rews_t = torch.stack(rews, dim=0)  # [H, B]

    # critic values along an imagined trajectory
    vals = []
    for t in range(H):
        v_t = dreamer.critic(zs_t[t], hs_t[t]).squeeze(-1)  # [B]
        vals.append(v_t)
    vals = torch.stack(vals, dim=0)  # [H, B]

    # lambda-returns as targets
    targets = lambda_returns(rews_t, vals, gamma, lam)
    L_actor = actor_loss(targets)

    hs_t_detached = [h.detach() for h in hs_t]
    zs_t_detached = [z.detach() for z in zs_t]
    
    L_critic = critic_loss(dreamer, hs_t_detached, zs_t_detached, targets.detach())

    actor_opt.zero_grad()
    L_actor.backward(retain_graph=True) 
    actor_opt.step()

    critic_opt.zero_grad()
    L_critic.backward()
    critic_opt.step()

    return L_actor.item() + L_critic.item(), L_actor.item(), L_critic.item()


### Training!

In [17]:
import gymnasium as gym
import numpy as np

class ReplayBuffer:
    def __init__(self, capacity, obs_size, action_size):
        self.capacity = capacity
        self.obs_buf = np.zeros((capacity, obs_size), dtype=np.float32)
        self.act_buf = np.zeros((capacity, action_size), dtype=np.float32)
        self.rew_buf = np.zeros((capacity,), dtype=np.float32)
        self.done_buf = np.zeros((capacity,), dtype=np.float32)
        self.ptr = 0
        self.size = 0

    def add(self, obs, act, rew, done):
        self.obs_buf[self.ptr] = obs
        self.act_buf[self.ptr] = act
        self.rew_buf[self.ptr] = rew
        self.done_buf[self.ptr] = done
        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample_sequences(self, seq_len, batch_size):
        obs_seqs, act_seqs, rew_seqs = [], [], []
        
        while len(obs_seqs) < batch_size:
            idx = np.random.randint(0, self.size - seq_len)
            
            # check if an episode ends anywhere in this sequence 
            if np.any(self.done_buf[idx : idx + seq_len - 1]):
                continue
                
            obs_seqs.append(self.obs_buf[idx:idx+seq_len])    
            act_seqs.append(self.act_buf[idx:idx+seq_len])    
            rew_seqs.append(self.rew_buf[idx:idx+seq_len])    
            
        obs_seqs = torch.tensor(np.stack(obs_seqs, axis=1))   
        act_seqs = torch.tensor(np.stack(act_seqs, axis=1))   
        rew_seqs = torch.tensor(np.stack(rew_seqs, axis=1))   
        return obs_seqs, act_seqs, rew_seqs


In [18]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(device)

env = gym.make("MountainCarContinuous-v0")
obs_size = env.observation_space.shape[0]
action_size = env.action_space.shape[0]

# params
latent_size = 32
hidden_size = 200
obs_embed_dim = 32
gamma = 0.99
lam = 0.95

dreamer = Dreamer(
    obs_size=obs_size,
    latent_size=latent_size,
    hidden_size=hidden_size,
    action_size=action_size,
).to(device)

dreamer.obs_encoder = ObsEncoder(obs_size, obs_embed_dim).to(device)
dreamer.rssm = RSSM(latent_size, action_size, hidden_size, obs_embed_dim).to(device)

wm_opt = torch.optim.Adam(
    list(dreamer.rssm.parameters()) +
    list(dreamer.obs_encoder.parameters()) +
    list(dreamer.reward_decoder.parameters()) +
    list(dreamer.obs_decoder.parameters()),
    lr=3e-4,
)

actor_opt = torch.optim.Adam(dreamer.actor.parameters(), lr=1e-4)
critic_opt = torch.optim.Adam(dreamer.critic.parameters(), lr=3e-4)

replay = ReplayBuffer(
    capacity=100_000,
    obs_size=obs_size,
    action_size=action_size,
)


mps


In [19]:
def select_action(dreamer, obs, prev_action, h, z, random_prob=0.1):
    obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    o_emb = dreamer.obs_encoder(obs_t)
    
    if h is None:
        h, z = dreamer.rssm.init_state(batch_size=1, device=device)
        prev_action = np.zeros(dreamer.action_size)
        
    a_prev_t = torch.tensor(prev_action, dtype=torch.float32, device=device).unsqueeze(0)

    # update the agent's mental state
    h, z, _, _ = dreamer.rssm.observe_step(h, z, a_prev_t, o_emb)
    
    if np.random.rand() < random_prob:
        a = np.random.uniform(env.action_space.low, env.action_space.high, size=(dreamer.action_size,))
    else:
        a_t = dreamer.actor(z, h)
        a = a_t.detach().cpu().numpy()[0]
        a = np.clip(a, env.action_space.low, env.action_space.high)
        
    return a, h, z

In [20]:
num_steps = 50000
steps_per_update = 50
batch_len = 50
batch_size = 32
H = 15

obs, _ = env.reset()
h = None
z = None
prev_action = np.zeros(action_size)

for step in range(num_steps):
    action, h, z = select_action(dreamer, obs, prev_action, h, z, random_prob=0.3)
    next_obs, reward, terminated, truncated, _ = env.step(action)
    shaped_reward = reward + 100.0 * abs(next_obs[1])
    
    done = terminated or truncated
    replay.add(obs, action, shaped_reward, float(done))

    obs = next_obs
    prev_action = action

    if done:
        obs, _ = env.reset()
        h = None
        z = None
        prev_action = np.zeros(action_size)

    if replay.size > batch_len + batch_size and step % steps_per_update == 0:
        for _ in range(5):
            wm_loss, rew_l, kl_l = train_world_model_step(
                dreamer, wm_opt, replay,
                batch_len=batch_len, batch_size=batch_size,
                device=device, beta=1.0,
            )

        for _ in range(5):
            beh_loss, act_l, crt_l = train_behavior_step(
                dreamer, actor_opt, critic_opt, replay,
                H=H, device=device, gamma=gamma, lam=lam,
            )

        if replay.size > batch_len + batch_size and step % steps_per_update == 0:
            print(f"step {step} | wm_loss={wm_loss:.3f} | beh_loss={beh_loss:.3f}")



step 100 | wm_loss=3.102 | beh_loss=-0.396
step 150 | wm_loss=3.116 | beh_loss=-0.409
step 200 | wm_loss=3.166 | beh_loss=-0.296
step 250 | wm_loss=3.107 | beh_loss=0.300
step 300 | wm_loss=3.123 | beh_loss=0.437
step 350 | wm_loss=3.097 | beh_loss=0.063
step 400 | wm_loss=3.122 | beh_loss=0.069
step 450 | wm_loss=3.113 | beh_loss=0.120
step 500 | wm_loss=3.102 | beh_loss=0.245
step 550 | wm_loss=3.096 | beh_loss=0.287
step 600 | wm_loss=3.099 | beh_loss=0.218
step 650 | wm_loss=3.091 | beh_loss=0.218
step 700 | wm_loss=3.096 | beh_loss=0.101
step 750 | wm_loss=3.074 | beh_loss=-0.012
step 800 | wm_loss=3.081 | beh_loss=-0.241
step 850 | wm_loss=3.124 | beh_loss=-0.151
step 900 | wm_loss=3.118 | beh_loss=-0.080
step 950 | wm_loss=3.091 | beh_loss=0.058
step 1000 | wm_loss=3.325 | beh_loss=0.678
step 1050 | wm_loss=3.589 | beh_loss=0.724
step 1100 | wm_loss=3.196 | beh_loss=0.484
step 1150 | wm_loss=3.344 | beh_loss=0.149
step 1200 | wm_loss=3.414 | beh_loss=0.578
step 1250 | wm_loss=3.

In [ ]:
def watch_dreamer(dreamer, device, episodes=1, render_mode="human"):
    env = gym.make("MountainCarContinuous-v0", render_mode=render_mode)
    dreamer.eval()

    try: 
        for ep in range(episodes):
            obs, _ = env.reset()
            h, z = None, None
            total_reward = 0.0
            done = False

            prev_action = np.zeros(dreamer.action_size)

            while not done:
                obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                o_emb = dreamer.obs_encoder(obs_t)

                if h is None:
                    h, z = dreamer.rssm.init_state(batch_size=1, device=device)

                a_prev_t = torch.tensor(prev_action, dtype=torch.float32, device=device).unsqueeze(0)
                h, z, _, _ = dreamer.rssm.observe_step(h, z, a_prev_t, o_emb)

                a_t = dreamer.actor(z, h)
                action = a_t.detach().cpu().numpy()[0]
                action = np.clip(action, env.action_space.low, env.action_space.high)

                obs, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                total_reward += reward
                
                prev_action = action

            print(f"Episode {ep+1}: return = {total_reward:.2f}")

    except Exception as e:
        print(f"An error occurred: {e}")
        
    finally:
        print("Closing the environment window...")
        env.close()

watch_dreamer(dreamer, device, episodes=1, render_mode="human")


Episode 1: return = 84.97
Closing the environment window...


: 